In [ ]:
%load_ext autoreload
%autoreload 2
from __future__ import annotations
import os
import cProfile
import pstats

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from memory_profiler import memory_usage
from time import perf_counter

import gudhi
from gudhi import CubicalComplex
import cripser
import dionysus as dion

from topo.persistence_1d import find_extrema_in_timeseries, compute_1d_sublevel_persistence, compute_batch_sublevel_persistence, \
    StreamingSublevelPersistence, BatchStreamingSublevelPersistence
from topo.utils import DataGenerator1D, DataGenerator2D, MemoryUtils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
"💾 memory cleanup"
MemoryUtils.clear_jupyter_state()

MemoryUtils.clear_runtime_memory()

MemoryUtils.clear_compilation_cache()


In [ ]:
"[1d or 2d] 📊 Load real datasets"
df       = pd.read_csv("datasets/1D/NVidia_stock_history.csv")
y_vals_t = torch.tensor(df["Close"].values, dtype=torch.float32)#, device=device)

y_vals_flat = y_vals_t.repeat(1, 200).flatten()
y_vals = F.avg_pool1d(y_vals_flat.unsqueeze(0).unsqueeze(0), kernel_size=50, stride=1,).squeeze()
print(y_vals.shape, y_vals_t.shape)

# df     = pd.read_csv("../public_datasets/2D/tabular/longterm_weather/longterm_weather.csv")
# y_vals = torch.tensor(df["rh"].values, dtype=torch.float32)#, device=device)
y_new   = y_vals[:200]
y_new2  = y_vals[:200]
y_vals.shape
num_timesteps = y_vals.shape[0]
num_timesteps_new = num_timesteps
end_point = num_timesteps - 1


In [ ]:
"✅ [1D] setup, function-based"
matplotlib.rcParams['text.usetex'] = False

start_point   = 0
num_timesteps = 800_000
end_point     = num_timesteps//1
y_vals        = DataGenerator1D.make_yvals_1d(num_timesteps, start_point, end_point)

# num_timesteps_new = 100_000
# start_point_new   = end_point + 1
# end_point_new     = start_point_new+num_timesteps_new
# y_new = DataGenerator1D.make_yvals_1d(num_timesteps_new, start_point_new, end_point_new)

# num_timesteps_new2 = 100_000
# start_point_new2   = end_point_new + 1
# end_point_new2     = start_point_new2 + num_timesteps_new2
# y_new2 = DataGenerator1D.make_yvals_1d(num_timesteps_new2, start_point_new2, end_point_new2)


In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# household power
individual_household_electric_power_consumption = fetch_ucirepo(id=235)
X = individual_household_electric_power_consumption.data.features
y = individual_household_electric_power_consumption.data.targets

# power
appliances_energy_prediction = fetch_ucirepo(id=374) 
X = appliances_energy_prediction.data.features 
y = appliances_energy_prediction.data.targets 



In [ ]:
"🪑 [1D] benchmarks"

# # ===== 🪷 Gudhi =====
# start_gudhi = perf_counter()
# cc = CubicalComplex(dimensions=y_vals.shape, top_dimensional_cells=y_vals.numpy().flatten())
# cc.compute_persistence(homology_coeff_field=2) # default is 11, but 2 is faster for Z/2Z coefficients
# # pairs = cc.persistence()  # (dimension, (birth, death))

# # gudhi.plot_persistence_diagram(cc.persistence())
# # gudhi.plot_persistence_barcode(cc.persistence())

# # plt.figure()
# # plt.plot(x_vals.numpy(), y_vals.numpy())
# # plt.show()

# endtime_gudhi = perf_counter()
# print(f"🧮 [🪷 gudhi] persistence of {num_timesteps} y vals in {endtime_gudhi - start_gudhi:.4f} s")

# def bench_gudhi():
#     cc = CubicalComplex(dimensions=y_vals.shape, top_dimensional_cells=y_vals.numpy().flatten())
#     return cc.compute_persistence(homology_coeff_field=2) # default is 11, but 2 is faster for Z/2Z coefficients
# peak_mem_gudhi = max(memory_usage(bench_gudhi))
# print(f"💾 [🪷 gudhi] Peak RAM: {peak_mem_gudhi:.2f} MiB")

# start_gudhi2 = perf_counter()
# cc2 = CubicalComplex(dimensions=y_new.shape, top_dimensional_cells=y_new.numpy().flatten())
# cc2.compute_persistence(homology_coeff_field=2)
# # pairs2 = cc2.persistence()
# endtime_gudhi2 = perf_counter()
# print(f"⛓️‍💥 [🪷 gudhi] stream persistence of {num_timesteps_new} y vals in {endtime_gudhi2 - start_gudhi2:.4f} s")


# ===== ⚡️ cripser =====
arr             = y_vals.numpy()  # keep as 1D array, no need to flatten
start_cripser   = perf_counter()

# prof = cProfile.Profile()
# prof.enable()
ph              = cripser.compute_ph(arr, maxdim=0)
# prof.disable()
# stats = pstats.Stats(prof)
# stats.sort_stats("cumtime")
# stats.print_stats(30)

endtime_cripser = perf_counter()
print(f"🧮 [⚡️ cripser] persistence of {num_timesteps} y vals in {endtime_cripser - start_cripser:.4f} s")

# def bench_cripser():
#     return cripser.compute_ph(arr, maxdim=0)
# peak_mem_cripser = max(memory_usage(bench_cripser))

peak_mem_cripser = max(memory_usage(lambda: cripser.compute_ph(arr, maxdim=0)))
print(f"💾 [⚡️ cripser] Peak RAM: {peak_mem_cripser:.2f} MiB")

h0_1d_cripser = ph[ph[:, 0] == 0][:, 1:3]
print(f"CRiPSER 1D Layout -> H0: {len(h0_1d_cripser)} pairs")


# ===== 🦕 Dionysus =====
# start_dionysus = perf_counter()
# f    = dion.fill_freudenthal(y_vals.numpy())
# m    = dion.homology_persistence(f)
# # dgms = dion.init_diagrams(m, f)
# endtime_dionysus = perf_counter()
# print(f"🧮 [🦕 dionysus] persistence of {num_timesteps} y vals in {endtime_dionysus - start_dionysus:.4f} s")

# def bench_dionysus():
#     return dion.homology_persistence(dion.fill_freudenthal(y_vals.numpy()))
# peak_mem_dionysus = max(memory_usage(bench_dionysus))
# print(f"💾 [🦕 dionysus] Peak RAM: {peak_mem_dionysus:.2f} MiB")


In [ ]:
"⏰ [1D] gpu + numba"

# 1. INITIAL BASELINE (Compute & View Pairs)
torch.cuda.synchronize() if device.type == 'cuda' else None
starttime_gpu = perf_counter()

# prof  = cProfile.Profile()
# prof.enable()
keypoint_idx, keypoint_types = find_extrema_in_timeseries(y_vals, device=y_vals.device)
birth_death_pairs_list       = compute_1d_sublevel_persistence(y_vals, keypoint_idx, keypoint_types)
# prof.disable()
# stats = pstats.Stats(prof)
# stats.sort_stats("cumtime")
# stats.print_stats(30)

torch.cuda.synchronize() if device.type == 'cuda' else None
endtime_gpu   = perf_counter()
print(f"🧮 [😎 ours] persistence of {num_timesteps} yvals in {endtime_gpu - starttime_gpu:.4f}s")

# RAM
def bench_gpu():
    kp_idx, kp_types = find_extrema_in_timeseries(y_vals, device=y_vals.device)
    return compute_1d_sublevel_persistence(y_vals, kp_idx, kp_types)
peak_mem = max(memory_usage(bench_gpu))
print(f"💾 Peak RAM: {peak_mem:.2f} MiB")

# GPU VRAM (separate, additional cost)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats() if device.type == 'cuda' else None
kp_idx, kp_types = find_extrema_in_timeseries(y_vals, device=y_vals.device)
compute_1d_sublevel_persistence(y_vals, kp_idx, kp_types)
torch.cuda.synchronize() if device.type == 'cuda' else None
print(f"💾 Peak VRAM: {torch.cuda.max_memory_allocated()/1024**2:.2f} MiB")

# 2. INITIALIZE STREAMING PIPELINE (Allocate Max Capacity Upfront Once)
total_capacity = len(y_vals) + len(y_new) + len(y_new2) # set total capacity to safely fit everything
online_tda     = StreamingSublevelPersistence(y_vals, keypoint_idx, keypoint_types, max_capacity=total_capacity)

# 3. STREAM CHUNK 1 (y_new)
stream_keypoint_idx, stream_keypoint_types = find_extrema_in_timeseries(y_new, device=y_new.device)

# Guard condition directly inside your notebook runner cell
if stream_keypoint_idx.numel() == 0:
    print(f"ℹ️ stream chunk 1 has no topology features (flat line). Skipping state update.")
    new_birth_death_pairs_list = []
else:
    new_birth_death_pairs_list = online_tda.update_filtration(
        new_y_chunk=y_new,
        stream_keypoint_idx=stream_keypoint_idx,
        stream_keypoint_types=stream_keypoint_types)
endtime_gpu2 = perf_counter()
print(f"⛓️‍💥 stream chunk 1 persistence of {len(y_new)} yvals in {endtime_gpu2 - endtime_gpu:.4f}s")

# 4. STREAM CHUNK 2 (y_new2)
stream_keypoint_idx2, stream_keypoint_types2 = find_extrema_in_timeseries(y_new2, device=y_new2.device)

# Guard condition directly inside your notebook runner cell
if stream_keypoint_idx2.numel() == 0:
    print(f"ℹ️ stream chunk 2 has no topology features (flat line). Skipping state update.")
    new_birth_death_pairs_list2 = []
else:
    new_birth_death_pairs_list2 = online_tda.update_filtration(
        new_y_chunk=y_new2,
        stream_keypoint_idx=stream_keypoint_idx2,
        stream_keypoint_types=stream_keypoint_types2)
endtime_gpu3 = perf_counter()
print(f"⛓️‍💥 stream chunk 2 persistence of {len(y_new2)} yvals in {endtime_gpu3 - endtime_gpu2:.4f}s")


In [ ]:
"[👍 correct] our method 1d"
from topo.persistence_1d_correct import (
    PrefixStreamingPersistence1D,
    compute_1d_streamed_prefix_final,
    compute_1d_prefix_diagrams_by_chunks,
    compute_1d_prefix_diagrams_by_full_recompute,
    print_1d_streaming_work,
    print_1d_topological_activity,
)

n_trials = 3
m_chunks_1d = 10
y_vals_np = y_vals.detach().cpu().numpy().astype('float32') if hasattr(y_vals, 'detach') else np.asarray(y_vals, dtype='float32')
chunk_len_1d = max(1, len(y_vals_np) // m_chunks_1d)

print_1d_streaming_work("Correct full-array", len(y_vals_np))
print_1d_topological_activity("Correct full-array", y_vals_np)

print(f"\n==================== 1D WAY A: CORRECT FULL-ARRAY PREFIX (3x Trials) ====================")

full_1d_times = []
full_1d_mems  = []
full_1d_vms   = []

def bench_1d_correct_full():
    return compute_1d_streamed_prefix_final(y_vals_np, chunk_length=len(y_vals_np))

_ = bench_1d_correct_full()
MemoryUtils.clear_memory()

for trial in range(n_trials):
    MemoryUtils.clear_memory()
    start_full_1d = perf_counter()
    h0_1d_full = compute_1d_streamed_prefix_final(y_vals_np, chunk_length=len(y_vals_np))
    end_full_1d = perf_counter()

    full_1d_times.append(end_full_1d - start_full_1d)
    full_1d_vms.append(MemoryUtils.get_vms_mib())
    full_1d_mems.append(max(memory_usage(bench_1d_correct_full)))

    if trial == n_trials - 1:
        h0_1d_full_len = len(h0_1d_full)
    del h0_1d_full

print(f"\n🧮 [correct 1D full-array] Time:  {np.mean(full_1d_times):.4f} s ± {np.std(full_1d_times):.4f} s")
print(f"💾 [correct 1D full-array] Peak RAM:      {np.mean(full_1d_mems):.2f} MiB ± {np.std(full_1d_mems):.2f} MiB")
print(f"💾 [correct 1D full-array] Virtual (VMS): {np.mean(full_1d_vms):.2f} MiB ± {np.std(full_1d_vms):.2f} MiB")
print(f"Correct 1D Full Layout -> H0: {h0_1d_full_len} pairs")
print(f" & full& {np.mean(full_1d_times):.3f}$\pm${np.std(full_1d_times):.3f} & {np.mean(full_1d_mems):.2f}$\pm${np.std(full_1d_mems):.2f} \\")

print_1d_streaming_work("Correct 1D streamed", len(y_vals_np), chunk_length=chunk_len_1d)

print(f"\n==================== 1D WAY B0: FULL RECOMPUTE EVERY PREFIX (3x Trials) ====================")

recompute_1d_times = []
recompute_1d_mems  = []

def bench_1d_full_recompute_prefixes():
    return compute_1d_prefix_diagrams_by_full_recompute(y_vals_np, chunk_length=chunk_len_1d)[-1]

_ = bench_1d_full_recompute_prefixes()
MemoryUtils.clear_memory()

for trial in range(n_trials):
    MemoryUtils.clear_memory()
    start_recompute_1d = perf_counter()
    h0_1d_recompute = compute_1d_prefix_diagrams_by_full_recompute(y_vals_np, chunk_length=chunk_len_1d)[-1]
    end_recompute_1d = perf_counter()
    recompute_1d_times.append(end_recompute_1d - start_recompute_1d)
    recompute_1d_mems.append(max(memory_usage(bench_1d_full_recompute_prefixes)))
    if trial == n_trials - 1:
        h0_1d_recompute_len = len(h0_1d_recompute)
    del h0_1d_recompute

print(f"\n🧮 [1D full recompute prefixes] Time: {np.mean(recompute_1d_times):.4f} s ± {np.std(recompute_1d_times):.4f} s")
print(f"💾 [1D full recompute prefixes] Peak RAM: {np.mean(recompute_1d_mems):.2f} MiB ± {np.std(recompute_1d_mems):.2f} MiB")
print(f"Full Recompute Prefix Layout -> H0: {h0_1d_recompute_len} pairs")
print_1d_topological_activity("Correct 1D streamed", y_vals_np, chunk_length=chunk_len_1d)

print(f"\n==================== 1D WAY B: CORRECT PREFIX STREAMING (3x Trials) ====================")

stream_1d_times = []
stream_1d_mems  = []
stream_1d_vms   = []

def bench_1d_correct_streaming():
    return compute_1d_prefix_diagrams_by_chunks(y_vals_np, chunk_length=chunk_len_1d)[-1]

_ = bench_1d_correct_streaming()
MemoryUtils.clear_memory()

for trial in range(n_trials):
    MemoryUtils.clear_memory()
    start_stream_1d = perf_counter()

    engine_1d = PrefixStreamingPersistence1D()
    for chunk_idx, start in enumerate(range(0, len(y_vals_np), chunk_len_1d)):
        chunk = y_vals_np[start:start + chunk_len_1d]
        engine_1d.update_chunk(chunk)
        h0_1d_prefix = engine_1d.current_exact_prefix_diagram()
        if trial == 0:
            state = engine_1d.stored_state_summary()
            print(f"  ↳ [Chunk {chunk_idx + 1}] Length {len(chunk)} | prefix H0: {len(h0_1d_prefix)} | stored edges: {state['edges_stored']}")

    end_stream_1d = perf_counter()
    stream_1d_times.append(end_stream_1d - start_stream_1d)
    stream_1d_vms.append(MemoryUtils.get_vms_mib())
    stream_1d_mems.append(max(memory_usage(bench_1d_correct_streaming)))

    if trial == n_trials - 1:
        h0_1d_stream_len = len(h0_1d_prefix)
    del engine_1d, h0_1d_prefix

print(f"\n🧮 [correct 1D prefix streaming] Time:  {np.mean(stream_1d_times):.4f} s ± {np.std(stream_1d_times):.4f} s")
print(f"💾 [correct 1D prefix streaming] Peak RAM:      {np.mean(stream_1d_mems):.2f} MiB ± {np.std(stream_1d_mems):.2f} MiB")
print(f"💾 [correct 1D prefix streaming] Virtual (VMS): {np.mean(stream_1d_vms):.2f} MiB ± {np.std(stream_1d_vms):.2f} MiB")
print(f" & {m_chunks_1d}& {np.mean(stream_1d_times):.3f}$\pm${np.std(stream_1d_times):.3f} & {np.mean(stream_1d_mems):.2f}$\pm${np.std(stream_1d_mems):.2f} \\")

print(f"\nCorrect 1D Streamed Layout -> H0: {h0_1d_stream_len} pairs")
print(f"Prefix speedup vs full recompute: {np.mean(recompute_1d_times) / np.mean(stream_1d_times):.2f}x")
print(f"Full-vs-Stream Match? H0: {'✅' if h0_1d_full_len == h0_1d_stream_len else '❌'}")

try:
    ph_1d_cripser = cripser.compute_ph(y_vals_np, maxdim=0)
    h0_1d_cripser = ph_1d_cripser[ph_1d_cripser[:, 0] == 0][:, 1:3]
    print(f"CRiPSER 1D Layout -> H0: {len(h0_1d_cripser)} pairs")
except Exception as exc:
    print(f"CRiPSER 1D comparison skipped: {exc}")

MemoryUtils.clear_memory()


In [ ]:
"🥋 [1D] MASTER (batching + streaming)"

starttime_master = perf_counter()

# --- STEP 1: Slice your base data into a "Baseline" block and a "Streaming Chunk" block ---
baseline_cutoff = int(len(y_vals) * 0.8)

y_vals_baseline = y_vals[:baseline_cutoff]
y_vals_chunk    = y_vals[baseline_cutoff:]

# --- STEP 2: Build the Initial Baseline 2D Matrix (N = 5) ---
y_batch = torch.stack([
    y_vals_baseline, 
    1.2 * y_vals_baseline, 
    -0.8 * y_vals_baseline, 
    -0.4 + 2.1 * y_vals_baseline, 
    0.2 - 0.1 * y_vals_baseline])

# --- STEP 3: Build your incoming streaming 2D Matrix (N = 5) ---
y_chunk_matrix = torch.stack([
    y_vals_chunk, 
    1.2 * y_vals_chunk, 
    -0.8 * y_vals_chunk, 
    -0.4 + 2.1 * y_vals_chunk, 
    0.2 - 0.1 * y_vals_chunk])

# --- STEP 4: Run the pipeline ---
idx_list, types_list = [], []
for i in range(y_batch.shape[0]):
    idx, types = find_extrema_in_timeseries(y_batch[i], device=y_batch.device)
    idx_list.append(idx)
    types_list.append(types)

# Spin up the global 2D tracking system
batch_online_tda = BatchStreamingSublevelPersistence(y_batch, idx_list, types_list)

# Extract chunk keypoints
new_idx_list, new_types_list = [], []
for i in range(y_chunk_matrix.shape[0]):
    idx, types = find_extrema_in_timeseries(y_chunk_matrix[i], device=y_chunk_matrix.device)
    new_idx_list.append(idx)
    new_types_list.append(types)

# Process the new chunk instantaneously across all 5 channels!
new_batch_pairs = batch_online_tda.append_batch_stream(y_chunk_matrix, new_idx_list, new_types_list)

endtime_master = perf_counter()
print(f"🧮 Persistence of {len(y_vals_chunk)} streaming yvals processed across {y_batch.shape[0]} channels in {endtime_master - starttime_master:.4f}s")

# # --- STEP 5: Verification Printout (Sanity Check) ---
# print("\n🔍 Verification Lookups:")
# for i, single_series_pairs in enumerate(new_batch_pairs):
#     print(f"  Channel #{i}: Found {len(single_series_pairs)} new pairs.")
#     if len(single_series_pairs) > 0:
#         b_idx, d_idx = single_series_pairs[0]  # Check first pair discovered
#         birth_val = batch_online_tda.history_values[i, b_idx]
#         death_val = batch_online_tda.history_values[i, d_idx]
#         print(f"    └─ Sample Pair -> Indices: ({b_idx}, {d_idx}) | Values: ({birth_val:2.3f}, {death_val:2.3f})")



In [ ]:
"[1D] speedup comparer"
N_points = end_point
K        = len(keypoint_idx)

cripser_ops = 2*N_points*np.log2(2*N_points) + 6*N_points
our_ops     = 2*N_points + K*np.log2(K) + 4*K

print(f"{N_points} points, {K} keypoints, {K/N_points*100:.2f}% are keypoints")
print(f"#ops: Cripser={cripser_ops:.2e}, Ours={our_ops:.2e}")

theoretical_speedup = cripser_ops / our_ops
actual_speedup = (endtime_cripser - start_cripser) / (endtime_gpu - starttime_gpu)
speedup_ratio = actual_speedup/theoretical_speedup
speedup_emoji = "🚀" if actual_speedup > theoretical_speedup else "🐌"
print(f"Speedup: theoretical={theoretical_speedup:.2f}x actual={actual_speedup:.2f}x ratio={speedup_ratio:.2f} {speedup_emoji}")


In [ ]:
"📗 [2D] setup"
array_dimensions = (500, 500)
# arr_2d    = DataGenerator2D.generate_random_matrix(*array_dimensions, low=0.0, high=30)
arr_2d_patch    = DataGenerator2D.generate_patchy_matrix(*array_dimensions, device=device, min_val=0.0, max_val=10, smooth_radius=3)
arr_2d_donut    = DataGenerator2D.generate_donut_matrix(*array_dimensions, inner_radius=28, outer_radius=30)
arr_2d    = 1.5*arr_2d_donut + arr_2d_patch
arr_2d_np = arr_2d.numpy()

# # Separate H0 and H1 diagrams
# h0_cripser = ph_2d[ph_2d[:, 0] == 0][:, 1:3]  # Keeps only [Birth, Death]
# h1_cripser = ph_2d[ph_2d[:, 0] == 1][:, 1:3]  # Keeps only [Birth, Death]

# # Print summary to verify profiles
# print(f"CRiPSER # H0: {len(h0_cripser)}, # H1: {len(h1_cripser)}")
# # print("First few H1 pairs:\n", h1_cripser[:3])

plt.imshow(arr_2d, aspect="equal", cmap="viridis")
# plt.imshow(arr_2d[2*len(arr_2d)//3:, :], aspect="equal", cmap="viridis")
plt.axis("off")
plt.colorbar()
plt.show()


In [ ]:
"🪑 [2D] benchmarks"

n_trials  = 3
arr_2d_np = arr_2d.detach().cpu().numpy().astype('float32')
MemoryUtils.clear_memory()

print(f"\n============== RUNNING BASELINE BENCHMARKS (3x Trials) ===============")

# ⚡️ CRIPSER
def run_cripser():
    return cripser.compute_ph(arr_2d_np, maxdim=1)

cripser_times = []
cripser_mems  = []

for trial in range(n_trials):
    MemoryUtils.clear_memory()
    start_cripser_2d   = perf_counter()
    ph_2d              = run_cripser()
    endtime_cripser_2d = perf_counter()
    cripser_times.append(endtime_cripser_2d - start_cripser_2d)
    peak_mem_cripser   = max(memory_usage(run_cripser))
    cripser_mems.append(peak_mem_cripser)
    # del ph_2d

print(f"🧮 [⚡️ cripser] Time:     {np.mean(cripser_times):.4f} s ± {np.std(cripser_times):.4f} s")
print(f"💾 [⚡️ cripser] Peak RAM: {np.mean(cripser_mems):.2f} MiB ± {np.std(cripser_mems):.2f} MiB")
print(f"CubicalRipser & {np.mean(cripser_times):.3f}$\\pm${np.std(cripser_times):.3f} & {np.mean(cripser_mems):.2f}$\\pm${np.std(cripser_mems):.2f} \\")
MemoryUtils.clear_memory()

h0_cripser = ph_2d[ph_2d[:, 0] == 0][:, 1:3]
h1_cripser = ph_2d[ph_2d[:, 0] == 1][:, 1:3]
print(f"CRiPSER Layout -> H0: {len(h0_cripser)} pairs | H1: {len(h1_cripser)} pairs")


# 🪷 GUDHI 
def run_gudhi():
    cc = CubicalComplex(dimensions=arr_2d_np.shape, top_dimensional_cells=arr_2d_np.flatten())
    cc.compute_persistence(homology_coeff_field=2) # default is 11, but 2 is faster for Z/2Z coefficients
    return cc

gudhi_times = []
gudhi_mems  = []

for trial in range(n_trials):
    MemoryUtils.clear_memory()
    start_gudhi_2d   = perf_counter()
    _                = run_gudhi()
    endtime_gudhi_2d = perf_counter()
    gudhi_times.append(endtime_gudhi_2d - start_gudhi_2d)
    peak_mem_gudhi   = max(memory_usage(run_gudhi))
    gudhi_mems.append(peak_mem_gudhi)

print(f"🧮 [🪷 gudhi] Time:     {np.mean(gudhi_times):.4f} s ± {np.std(gudhi_times):.4f} s")
print(f"💾 [🪷 gudhi] Peak RAM: {np.mean(gudhi_mems):.2f} MiB ± {np.std(gudhi_mems):.2f} MiB")
print(f"Gudhi & {np.mean(gudhi_times):.3f}$\\pm$ {np.std(gudhi_times):.3f} & {np.mean(gudhi_mems):.2f}$\\pm${np.std(gudhi_mems):.2f} \\")
MemoryUtils.clear_memory()

pairs_gudhi = _.persistence()
h0_gudhi = [p for p in pairs_gudhi if p[0] == 0]
h1_gudhi = [p for p in pairs_gudhi if p[0] == 1]
print(f"Gudhi Layout -> H0: {len(h0_gudhi)} pairs | H1: {len(h1_gudhi)} pairs")

# 3. Dionysus 2 Benchmark Setup
# def run_dionysus():
#     f = dion.fill_freudenthal(arr_2d_np)
#     return dion.homology_persistence(f)

# dionysus_times = []
# dionysus_mems  = []

# for trial in range(n_trials):
#     MemoryUtils.clear_memory()
#     start_dionysus    = perf_counter()
#     m                 = run_dionysus()
#     endtime_dionysus  = perf_counter()
#     dionysus_times.append(endtime_dionysus - start_dionysus)
#     peak_mem_dionysus = max(memory_usage(run_dionysus))
#     dionysus_mems.append(peak_mem_dionysus)
#     del m

# print(f"🧮 [🦕 dionysus] Time:     {np.mean(dionysus_times):.4f} s ± {np.std(dionysus_times):.4f} s")
# print(f"💾 [🦕 dionysus] Peak RAM: {np.mean(dionysus_mems):.2f} MiB ± {np.std(dionysus_mems):.2f} MiB")
# print("================================================================================\n")
# MemoryUtils.clear_memory()


In [ ]:
"our method 2d"
from topo.persistence_2d import compute_h0_h1_fast, _run_streaming_persistence, prepare_multi_chunk_wrapper
from topo.persistence_2d_correct import print_grid_work, print_adaptive_work

n_trials = 3
m_chunks = 10

print_grid_work("Old global / chunk-packed", arr_2d.shape)
print_adaptive_work("Topology-aware reference", arr_2d.detach().cpu().numpy().astype("float32"))

print(f"\n==================== WAY 1: GLOBAL (3x Trials) ====================")

global_times = []
global_mems  = []

def bench_2d():
    return compute_h0_h1_fast(arr_2d, device=device)

# warmup pass to eliminate initial JIT/numba compilation overhead
_ = bench_2d()
MemoryUtils.clear_memory()

for trial in range(n_trials):
    start_our_2d    = perf_counter()
    h0, h1          = compute_h0_h1_fast(arr_2d, device=device)
    endtime_our_2d  = perf_counter()
    global_times.append(endtime_our_2d - start_our_2d)
    peak_mem_global = max(memory_usage(bench_2d))
    global_mems.append(peak_mem_global)
    
    if trial == n_trials - 1:  # Save dimensions for reference report on last trial
        h0_len, h1_len = len(h0), len(h1)
    del h0, h1
    MemoryUtils.clear_memory()

print(f"🧮 [😎 ours global] Time:     {np.mean(global_times):.4f} s ± {np.std(global_times):.4f} s")
print(f"💾 [😎 ours global] Peak RAM: {np.mean(global_mems):.2f} MiB ± {np.std(global_mems):.2f} MiB")
# print(f"   ↳ # H0 pairs: {h0_len}, # H1 pairs: {h1_len}")
# print(f"Ours & {np.mean(global_times):.3f}$\\pm${np.std(global_times):.3f} & {np.mean(global_mems):.2f}$\\pm${np.std(global_mems):.2f} \\")

print_grid_work("Old simulated streaming", arr_2d.shape, chunk_rows=max(1, arr_2d.shape[0] // m_chunks))

print(f"\n==================== WAY 2: ROW-WISE CHUNK STREAMING (3x Trials) ====================")

stream_times = []
stream_mems  = []
stream_vms   = []

def bench_streaming():
    data_package = prepare_multi_chunk_wrapper(arr_2d, m_chunks=m_chunks)
    return _run_streaming_persistence(data_package)

# Warmup pass to eliminate JIT compilation on streaming loops
_ = bench_streaming()
MemoryUtils.clear_memory()

for trial in range(n_trials):
    MemoryUtils.clear_memory()
    if trial == 0:
        print(f"📦 Simulating stream arrival of {m_chunks} horizontal cuts (Verbose logging on Trial 1)...")
    start_streaming_total = perf_counter()
    chunks                = torch.tensor_split(arr_2d, m_chunks, dim=0)
    packed_chunk_data     = []

    for idx, chunk in enumerate(chunks):
        start_chunk   = perf_counter()
        chunk_package = prepare_multi_chunk_wrapper(chunk, m_chunks=1)
        packed_chunk_data.append(chunk_package)
        end_chunk     = perf_counter()
        if trial == 0:
            print(f"  ↳ 🚰 [Chunk {idx+1}/{m_chunks}] Footprint {chunk.shape[0]}x{chunk.shape[1]} in {end_chunk - start_chunk:.4f}s")

    if trial == 0:
        print(f"⚙️ Executing interface stitching across boundaries and compiling final diagram...")
        
    data_package         = prepare_multi_chunk_wrapper(arr_2d, m_chunks=m_chunks)
    h0_stream, h1_stream = _run_streaming_persistence(data_package)
    end_streaming_total  = perf_counter()
    stream_times.append(end_streaming_total - start_streaming_total)
    stream_vms.append(MemoryUtils.get_vms_mib())
    peak_mem_stream      = max(memory_usage(bench_streaming))
    stream_mems.append(peak_mem_stream)
    
    if trial == n_trials - 1:
        h0_stream_len, h1_stream_len = len(h0_stream), len(h1_stream)
    del h0_stream, h1_stream, data_package, packed_chunk_data, chunks

print(f"\n🧮 [🚰 streaming total] Time:  {np.mean(stream_times):.4f} s ± {np.std(stream_times):.4f} s")
print(f"💾 [🚰 streaming] Peak RAM:      {np.mean(stream_mems):.2f} MiB ± {np.std(stream_mems):.2f} MiB")
print(f"💾 [🚰 streaming] Virtual (VMS): {np.mean(stream_vms):.2f} MiB ± {np.std(stream_vms):.2f} MiB")
print(f" & {m_chunks}& {np.mean(stream_times):.3f}$\\pm${np.std(stream_times):.3f} & {np.mean(stream_mems):.2f}$\\pm${np.std(stream_mems):.2f} \\\\")

# Quick verification verification check against fresh global run
h0_validate, h1_validate = compute_h0_h1_fast(arr_2d, device=device)
print(f"\nStreamed Layout -> H0: {h0_stream_len} pairs | H1: {h1_stream_len} pairs")
print(f"Match Success? H0: {'✅' if h0_stream_len == len(h0_validate) else '❌'} | H1: {'✅' if h1_stream_len == len(h1_validate) else '❌'}")

del h0_validate, h1_validate
MemoryUtils.clear_memory()


In [ ]:
"[👍 2D correct] our method"
from topo.persistence_2d import compute_h0_h1_fast
from topo.persistence_2d_correct import compute_streamed_by_rows, compute_exact_prefix_diagrams_by_rows, compute_2d_prefix_diagrams_by_full_recompute, IncrementalStreamingPersistence2D, print_grid_work, print_adaptive_work

n_trials = 3
m_chunks = 10
chunk_rows = max(1, arr_2d.shape[0] // m_chunks)
arr_2d_np = arr_2d.detach().cpu().numpy().astype('float32')

print_grid_work("Correct full-array", arr_2d_np.shape)
print_adaptive_work("Correct full-array", arr_2d_np)

print(f"\n==================== WAY 3A: CORRECT FULL-ARRAY CONSTRUCTION (3x Trials) ====================")

correct_full_times = []
correct_full_mems  = []
correct_full_vms   = []

def bench_correct_full():
    return compute_streamed_by_rows(arr_2d_np, chunk_rows=arr_2d_np.shape[0])

# warmup
_ = bench_correct_full()
MemoryUtils.clear_memory()

for trial in range(n_trials):
    MemoryUtils.clear_memory()
    start_correct_full = perf_counter()
    h0_full, h1_full = compute_streamed_by_rows(arr_2d_np, chunk_rows=arr_2d_np.shape[0])
    end_correct_full = perf_counter()

    correct_full_times.append(end_correct_full - start_correct_full)
    correct_full_vms.append(MemoryUtils.get_vms_mib())
    correct_full_mems.append(max(memory_usage(bench_correct_full)))

    if trial == n_trials - 1:
        h0_full_len, h1_full_len = len(h0_full), len(h1_full)
    del h0_full, h1_full

print(f"\n🧮 [correct full-array] Time:  {np.mean(correct_full_times):.4f} s ± {np.std(correct_full_times):.4f} s")
print(f"💾 [correct full-array] Peak RAM:      {np.mean(correct_full_mems):.2f} MiB ± {np.std(correct_full_mems):.2f} MiB")
print(f"💾 [correct full-array] Virtual (VMS): {np.mean(correct_full_vms):.2f} MiB ± {np.std(correct_full_vms):.2f} MiB")
print(f"Full Array Layout -> H0: {h0_full_len} pairs | H1: {h1_full_len} pairs")
print(f" & full& {np.mean(correct_full_times):.3f}$\pm${np.std(correct_full_times):.3f} & {np.mean(correct_full_mems):.2f}$\pm${np.std(correct_full_mems):.2f} \\")

print_grid_work("Correct row-streamed", arr_2d_np.shape, chunk_rows=chunk_rows)
print_adaptive_work("Correct row-streamed", arr_2d_np, chunk_rows=chunk_rows)

print(f"\n==================== WAY 3B0: FULL RECOMPUTE EVERY PREFIX (3x Trials) ====================")

recompute_2d_times = []
recompute_2d_mems  = []

def bench_2d_full_recompute_prefixes():
    return compute_2d_prefix_diagrams_by_full_recompute(arr_2d_np, chunk_rows=chunk_rows)[-1]

_ = bench_2d_full_recompute_prefixes()
MemoryUtils.clear_memory()

for trial in range(n_trials):
    MemoryUtils.clear_memory()
    start_recompute_2d = perf_counter()
    h0_2d_recompute, h1_2d_recompute = compute_2d_prefix_diagrams_by_full_recompute(arr_2d_np, chunk_rows=chunk_rows)[-1]
    end_recompute_2d = perf_counter()
    recompute_2d_times.append(end_recompute_2d - start_recompute_2d)
    recompute_2d_mems.append(max(memory_usage(bench_2d_full_recompute_prefixes)))
    if trial == n_trials - 1:
        h0_2d_recompute_len, h1_2d_recompute_len = len(h0_2d_recompute), len(h1_2d_recompute)
    del h0_2d_recompute, h1_2d_recompute

print(f"\n🧮 [2D full recompute prefixes] Time: {np.mean(recompute_2d_times):.4f} s ± {np.std(recompute_2d_times):.4f} s")
print(f"💾 [2D full recompute prefixes] Peak RAM: {np.mean(recompute_2d_mems):.2f} MiB ± {np.std(recompute_2d_mems):.2f} MiB")
print(f"Full Recompute Prefix Layout -> H0: {h0_2d_recompute_len} pairs | H1: {h1_2d_recompute_len} pairs")

print(f"\n==================== WAY 3B: CORRECT ROW-STREAMED CONSTRUCTION (3x Trials) ====================")

correct_stream_times = []
correct_stream_mems  = []
correct_stream_vms   = []

def bench_correct_streaming():
    return compute_exact_prefix_diagrams_by_rows(arr_2d_np, chunk_rows=chunk_rows)[-1]

# warmup
_ = bench_correct_streaming()
MemoryUtils.clear_memory()

for trial in range(n_trials):
    MemoryUtils.clear_memory()
    start_correct_stream = perf_counter()

    engine = IncrementalStreamingPersistence2D()
    for chunk_idx, start in enumerate(range(0, arr_2d_np.shape[0], chunk_rows)):
        chunk = arr_2d_np[start:start + chunk_rows]
        engine.update_chunk(chunk)
        h0_prefix, h1_prefix = engine.current_exact_prefix_diagrams()
        if trial == 0:
            state = engine.stored_state_summary()
            print(f"  ↳ [Chunk {chunk_idx + 1}] Footprint {chunk.shape[0]}x{chunk.shape[1]} | prefix H0: {len(h0_prefix)} | prefix H1: {len(h1_prefix)} | stored H0 edge blocks: {state['h0_component_edge_blocks']}")

    h0_correct, h1_correct = h0_prefix, h1_prefix
    end_correct_stream = perf_counter()

    correct_stream_times.append(end_correct_stream - start_correct_stream)
    correct_stream_vms.append(MemoryUtils.get_vms_mib())
    correct_stream_mems.append(max(memory_usage(bench_correct_streaming)))

    if trial == n_trials - 1:
        h0_correct_len, h1_correct_len = len(h0_correct), len(h1_correct)
    del engine, h0_correct, h1_correct

print(f"\n🧮 [correct streaming construction] Time:  {np.mean(correct_stream_times):.4f} s ± {np.std(correct_stream_times):.4f} s")
print(f"💾 [correct streaming construction] Peak RAM:      {np.mean(correct_stream_mems):.2f} MiB ± {np.std(correct_stream_mems):.2f} MiB")
print(f"💾 [correct streaming construction] Virtual (VMS): {np.mean(correct_stream_vms):.2f} MiB ± {np.std(correct_stream_vms):.2f} MiB")
print(f" & {m_chunks}& {np.mean(correct_stream_times):.3f}$\pm${np.std(correct_stream_times):.3f} & {np.mean(correct_stream_mems):.2f}$\pm${np.std(correct_stream_mems):.2f} \\")

h0_validate, h1_validate = compute_h0_h1_fast(arr_2d, device=device)
print(f"\nCorrect Streamed Layout -> H0: {h0_correct_len} pairs | H1: {h1_correct_len} pairs")
print(f"Prefix speedup vs full recompute: {np.mean(recompute_2d_times) / np.mean(correct_stream_times):.2f}x")
print(f"Match Success? H0: {'✅' if h0_correct_len == len(h0_validate) else '❌'} | H1: {'✅' if h1_correct_len == len(h1_validate) else '❌'}")
print(f"Full-vs-Stream Match? H0: {'✅' if h0_full_len == h0_correct_len else '❌'} | H1: {'✅' if h1_full_len == h1_correct_len else '❌'}")

del h0_validate, h1_validate
MemoryUtils.clear_memory()


In [ ]:
"📈 theoretical recycled-work scaling curves"
from topo.streaming_work_analysis import (
    construction_work_curve_1d,
    construction_work_curve_2d,
    final_work_ratio,
    topology_activity_summary_1d,
    topology_activity_summary_2d,
    print_topology_activity_summary,)

# Theoretical construction work; independent of exact scalar values.
y_vals_np_for_work    = y_vals.detach().cpu().numpy().astype('float32') if hasattr(y_vals, 'detach') else np.asarray(y_vals, dtype='float32')
arr_2d_shape_for_work = tuple(arr_2d.shape)

n_samples_work = len(y_vals_np_for_work)
n_rows_work, n_cols_work = arr_2d_shape_for_work
n_pixels_work = n_rows_work * n_cols_work

main_chunks = 10
chunk_counts = [1, 2, 5, 10, 20]
chunk_counts_2d = [c for c in chunk_counts if c <= n_rows_work]

main_chunk_len_1d = max(1, int(np.ceil(n_samples_work / main_chunks)))
main_chunk_rows_2d = max(1, int(np.ceil(n_rows_work / main_chunks)))

curve_1d = construction_work_curve_1d(n_samples_work, main_chunk_len_1d)
curve_2d = construction_work_curve_2d(n_rows_work, n_cols_work, main_chunk_rows_2d)
curve_2d_x_pixels = curve_2d['prefix_rows'] * n_cols_work

print(f"1D total construction-work ratio, rebuild / recycled incremental build: {final_work_ratio(curve_1d):.2f}x")
print(f"2D total construction-work ratio, rebuild / recycled incremental build: {final_work_ratio(curve_2d):.2f}x")
print("Work units are cubical cells/events constructed, not CPU instructions.")
print("1D work units = vertices + adjacent edges; 2D work units = vertices + grid edges + square faces.")

activity_1d = topology_activity_summary_1d(y_vals_np_for_work, chunk_length=main_chunk_len_1d)
activity_2d = topology_activity_summary_2d(arr_2d, chunk_rows=main_chunk_rows_2d)
print_topology_activity_summary("1D current dataset", activity_1d)
print_topology_activity_summary("2D current dataset", activity_2d)

fig, axes = plt.subplots(3, 2, figsize=(14, 13.0))

# 1) Main marginal method comparison.
axes[0, 0].plot(curve_1d['prefix_size'], curve_1d['full_recompute_per_step'], marker='o', label='rebuild current array')
axes[0, 0].plot(curve_1d['prefix_size'], curve_1d['recycled_incremental_per_step'], marker='o', label='recycled: new chunk only')
axes[0, 0].set_title('1D marginal construction work (10 chunks)')
axes[0, 0].set_xlabel('accumulated array size (# samples seen)')
axes[0, 0].set_ylabel('constructed cells/events for this chunk')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.25)

axes[0, 1].plot(curve_2d_x_pixels, curve_2d['full_recompute_per_step'], marker='o', label='rebuild current array')
axes[0, 1].plot(curve_2d_x_pixels, curve_2d['recycled_incremental_per_step'], marker='o', label='recycled: new rows only')
axes[0, 1].set_title('2D marginal construction work (10 chunks)')
axes[0, 1].set_xlabel('accumulated array size (# pixels seen)')
axes[0, 1].set_ylabel('constructed cells/events for this chunk')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.25)

# 2) Chunk-count sensitivity for marginal recycled work.
for n_chunks in chunk_counts:
    chunk_len = max(1, int(np.ceil(n_samples_work / n_chunks)))
    c = construction_work_curve_1d(n_samples_work, chunk_len)
    axes[1, 0].plot(c['prefix_size'], c['recycled_incremental_per_step'], marker='o', label=f'{n_chunks} chunks')
axes[1, 0].set_title('1D chunk-count sensitivity')
axes[1, 0].set_xlabel('accumulated array size (# samples seen)')
axes[1, 0].set_ylabel('incremental cells/events built this chunk')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25)

for n_chunks in chunk_counts_2d:
    chunk_rows = max(1, int(np.ceil(n_rows_work / n_chunks)))
    c = construction_work_curve_2d(n_rows_work, n_cols_work, chunk_rows)
    axes[1, 1].plot(c['prefix_rows'] * n_cols_work, c['recycled_incremental_per_step'], marker='o', label=f'{n_chunks} chunks')
axes[1, 1].set_title('2D chunk-count sensitivity')
axes[1, 1].set_xlabel('accumulated array size (# pixels seen)')
axes[1, 1].set_ylabel('incremental cells/events built this chunk')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.25)

# 3) Cumulative method comparison.
axes[2, 0].plot(curve_1d['prefix_size'], curve_1d['full_recompute_cumulative'], marker='o', label='rebuild cumulative')
axes[2, 0].plot(curve_1d['prefix_size'], curve_1d['recycled_cumulative'], marker='o', label='recycled cumulative')
axes[2, 0].plot(curve_1d['prefix_size'], curve_1d['saved_cumulative'], linestyle='--', marker='o', label='cumulative work avoided')
axes[2, 0].set_title('1D cumulative construction work (10 chunks)')
axes[2, 0].set_xlabel('accumulated array size (# samples seen)')
axes[2, 0].set_ylabel('total constructed cells/events so far')
axes[2, 0].legend()
axes[2, 0].grid(alpha=0.25)

axes[2, 1].plot(curve_2d_x_pixels, curve_2d['full_recompute_cumulative'], marker='o', label='rebuild cumulative')
axes[2, 1].plot(curve_2d_x_pixels, curve_2d['recycled_cumulative'], marker='o', label='recycled cumulative')
axes[2, 1].plot(curve_2d_x_pixels, curve_2d['saved_cumulative'], linestyle='--', marker='o', label='cumulative work avoided')
axes[2, 1].set_title('2D cumulative construction work (10 chunks)')
axes[2, 1].set_xlabel('accumulated array size (# pixels seen)')
axes[2, 1].set_ylabel('total constructed cells/events so far')
axes[2, 1].legend()
axes[2, 1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

# Final total work as chunk count changes.
chunk_counts_total_1d = np.array(chunk_counts, dtype=np.int64)
full_total_1d = []
recycled_total_1d = []
for n_chunks in chunk_counts_total_1d:
    chunk_len = max(1, int(np.ceil(n_samples_work / n_chunks)))
    c = construction_work_curve_1d(n_samples_work, chunk_len)
    full_total_1d.append(c['full_recompute_cumulative'][-1])
    recycled_total_1d.append(c['recycled_cumulative'][-1])

chunk_counts_total_2d = np.array(chunk_counts_2d, dtype=np.int64)
full_total_2d = []
recycled_total_2d = []
for n_chunks in chunk_counts_total_2d:
    chunk_rows = max(1, int(np.ceil(n_rows_work / n_chunks)))
    c = construction_work_curve_2d(n_rows_work, n_cols_work, chunk_rows)
    full_total_2d.append(c['full_recompute_cumulative'][-1])
    recycled_total_2d.append(c['recycled_cumulative'][-1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

axes[0].plot(chunk_counts_total_1d, full_total_1d, marker='o', label='rebuild every chunk')
axes[0].plot(chunk_counts_total_1d, recycled_total_1d, marker='o', label='recycled construction')
axes[0].set_title('1D final total construction work vs chunk count')
axes[0].set_xlabel('number of incoming chunks')
axes[0].set_ylabel('total constructed cells/events')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(chunk_counts_total_2d, full_total_2d, marker='o', label='rebuild every chunk')
axes[1].plot(chunk_counts_total_2d, recycled_total_2d, marker='o', label='recycled construction')
axes[1].set_title('2D final total construction work vs chunk count')
axes[1].set_xlabel('number of incoming chunks')
axes[1].set_ylabel('total constructed cells/events')
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:
"📊 actual recycled-prefix scaling metrics"
from topo.streaming_work_analysis import (
    actual_prefix_metrics_1d,
    actual_prefix_metrics_2d,
    actual_full_recompute_metrics_1d,
    actual_full_recompute_metrics_2d,
    actual_chunk_sensitivity_metrics_1d,
    actual_chunk_sensitivity_metrics_2d,
)

# Actual run metrics on the current y_vals and arr_2d objects.
y_vals_np_actual = y_vals.detach().cpu().numpy().astype('float32') if hasattr(y_vals, 'detach') else np.asarray(y_vals, dtype='float32')
arr_2d_np_actual = arr_2d.detach().cpu().numpy().astype('float32') if hasattr(arr_2d, 'detach') else np.asarray(arr_2d, dtype='float32')

m_chunks_actual = 10
chunk_len_actual_1d = max(1, len(y_vals_np_actual) // m_chunks_actual)
chunk_rows_actual_2d = max(1, arr_2d_np_actual.shape[0] // m_chunks_actual)

stream_1d = actual_prefix_metrics_1d(y_vals_np_actual, chunk_length=chunk_len_actual_1d)
recompute_1d = actual_full_recompute_metrics_1d(y_vals_np_actual, chunk_length=chunk_len_actual_1d)
stream_2d = actual_prefix_metrics_2d(arr_2d_np_actual, chunk_rows=chunk_rows_actual_2d)
recompute_2d = actual_full_recompute_metrics_2d(arr_2d_np_actual, chunk_rows=chunk_rows_actual_2d)
stream_2d_x_pixels = stream_2d['prefix_rows'] * arr_2d_np_actual.shape[1]
recompute_2d_x_pixels = recompute_2d['prefix_rows'] * arr_2d_np_actual.shape[1]

print(f"1D final -> streamed H0: {stream_1d['h0_pairs'][-1]} | recompute H0: {recompute_1d['h0_pairs'][-1]} | runtime ratio full/recycled: {recompute_1d['elapsed_sec'][-1] / stream_1d['elapsed_sec'][-1]:.2f}x")
print(f"2D final -> streamed H0/H1: {stream_2d['h0_pairs'][-1]}/{stream_2d['h1_pairs'][-1]} | recompute H0/H1: {recompute_2d['h0_pairs'][-1]}/{recompute_2d['h1_pairs'][-1]} | runtime ratio full/recycled: {recompute_2d['elapsed_sec'][-1] / stream_2d['elapsed_sec'][-1]:.2f}x")

# 1D: actual scaling against full recompute.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].plot(stream_1d['prefix_size'], stream_1d['elapsed_sec'], marker='o', label='recycled construction')
axes[0].plot(recompute_1d['prefix_size'], recompute_1d['elapsed_sec'], marker='o', label='rebuild each chunk')
axes[0].set_title('1D actual cumulative query time (10 chunks)')
axes[0].set_xlabel('current prefix length (# samples)')
axes[0].set_ylabel('cumulative seconds')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(stream_1d['prefix_size'], stream_1d['rss_mib'] - stream_1d['rss_mib'][0], marker='o', label='recycled construction')
axes[1].plot(recompute_1d['prefix_size'], recompute_1d['rss_mib'] - recompute_1d['rss_mib'][0], marker='o', label='rebuild each chunk')
axes[1].set_title('1D actual memory growth (10 chunks)')
axes[1].set_xlabel('current prefix length (# samples)')
axes[1].set_ylabel('RSS increase from first chunk (MiB)')
axes[1].legend()
axes[1].grid(alpha=0.25)

axes[2].plot(stream_1d['prefix_size'], stream_1d['h0_pairs'], marker='o', label='H0 pairs')
axes[2].plot(stream_1d['prefix_size'], stream_1d['critical_points'], linestyle='--', marker='o', label='critical points')
axes[2].set_title('1D actual topology counts (10 chunks)')
axes[2].set_xlabel('current prefix length (# samples)')
axes[2].set_ylabel('number of features/events')
axes[2].legend()
axes[2].grid(alpha=0.25)

plt.tight_layout()
plt.show()

# 2D: actual scaling against full recompute. Separate plot avoids 1D scale compression.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].plot(stream_2d_x_pixels, stream_2d['elapsed_sec'], marker='o', label='recycled construction')
axes[0].plot(recompute_2d_x_pixels, recompute_2d['elapsed_sec'], marker='o', label='rebuild each chunk')
axes[0].set_title('2D actual cumulative query time (10 chunks)')
axes[0].set_xlabel('accumulated image size (# pixels seen)')
axes[0].set_ylabel('cumulative seconds')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(stream_2d_x_pixels, stream_2d['rss_mib'] - stream_2d['rss_mib'][0], marker='o', label='recycled construction')
axes[1].plot(recompute_2d_x_pixels, recompute_2d['rss_mib'] - recompute_2d['rss_mib'][0], marker='o', label='rebuild each chunk')
axes[1].set_title('2D actual memory growth (10 chunks)')
axes[1].set_xlabel('accumulated image size (# pixels seen)')
axes[1].set_ylabel('RSS increase from first chunk (MiB)')
axes[1].legend()
axes[1].grid(alpha=0.25)

axes[2].plot(stream_2d_x_pixels, stream_2d['h0_pairs'], marker='o', label='H0 pairs')
axes[2].plot(stream_2d_x_pixels, stream_2d['h1_pairs'], marker='o', label='H1 pairs')
axes[2].plot(stream_2d_x_pixels, stream_2d['critical_pixels'], linestyle='--', marker='o', label='critical extrema/saddle-like pixels')
axes[2].set_title('2D actual topology counts (10 chunks; critical = extrema/saddle-like pixels)')
axes[2].set_xlabel('accumulated image size (# pixels seen)')
axes[2].set_ylabel('number of features/events')
axes[2].legend()
axes[2].grid(alpha=0.25)

plt.tight_layout()
plt.show()

# Chunk-count sensitivity: each line uses a different number of incoming chunks.
chunk_count_grid = [1, 2, 5, 10, 20]
chunk_count_grid_2d = [c for c in chunk_count_grid if c <= arr_2d_np_actual.shape[0]]

sensitivity_1d = actual_chunk_sensitivity_metrics_1d(y_vals_np_actual, chunk_count_grid)
sensitivity_2d = actual_chunk_sensitivity_metrics_2d(arr_2d_np_actual, chunk_count_grid_2d)

fig, axes = plt.subplots(2, 2, figsize=(13, 8.0))

for n_chunks, metrics in sensitivity_1d.items():
    axes[0, 0].plot(metrics['prefix_size'], metrics['elapsed_sec'], marker='o', label=f'{n_chunks} chunks')
    axes[0, 1].plot(metrics['prefix_size'], metrics['rss_mib'] - metrics['rss_mib'][0], marker='o', label=f'{n_chunks} chunks')
axes[0, 0].set_title('1D runtime sensitivity to number of chunks')
axes[0, 0].set_xlabel('current prefix length (# samples)')
axes[0, 0].set_ylabel('cumulative seconds')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.25)
axes[0, 1].set_title('1D memory sensitivity to number of chunks')
axes[0, 1].set_xlabel('current prefix length (# samples)')
axes[0, 1].set_ylabel('RSS increase from first chunk (MiB)')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.25)

for n_chunks, metrics in sensitivity_2d.items():
    axes[1, 0].plot(metrics['prefix_rows'] * arr_2d_np_actual.shape[1], metrics['elapsed_sec'], marker='o', label=f'{n_chunks} chunks')
    axes[1, 1].plot(metrics['prefix_rows'] * arr_2d_np_actual.shape[1], metrics['rss_mib'] - metrics['rss_mib'][0], marker='o', label=f'{n_chunks} chunks')
axes[1, 0].set_title('2D runtime sensitivity to number of chunks')
axes[1, 0].set_xlabel('accumulated image size (# pixels seen)')
axes[1, 0].set_ylabel('cumulative seconds')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25)
axes[1, 1].set_title('2D memory sensitivity to number of chunks')
axes[1, 1].set_xlabel('accumulated image size (# pixels seen)')
axes[1, 1].set_ylabel('RSS increase from first chunk (MiB)')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:
from skimage.segmentation import watershed
from skimage.feature import peak_local_max

# watershed gives basin labels directly
labels = watershed(arr_2d.cpu().numpy(), connectivity=1) # each unique label = one basin = one minimum


In [ ]:
x = np.linspace(0, 2, 300)
y = np.linspace(0, 1.2, 300)

X, Y = np.meshgrid(x, y)
Z    = np.sin(3 * np.pi * X) * (np.cos(2 * np.pi * Y))**2 * (X + Y + 1)

fig = plt.figure(figsize=(10, 8))
ax  = fig.add_subplot(111, projection="3d")

ax.plot_surface(X, Y, Z, cmap="terrain", edgecolor="none", alpha=0.9)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Surface Plot of Z = sin(3πx) cos^2(2πy) (x+y+1)")
plt.show()

arr_2d_np = Z.astype('float32')

cc = CubicalComplex(dimensions=arr_2d_np.shape,
                    top_dimensional_cells=arr_2d_np.flatten())
cc.compute_persistence(homology_coeff_field=2)

plt.figure(figsize=(6, 6))
gudhi.plot_persistence_diagram(cc.persistence())
plt.show()

plt.figure(figsize=(8, 4))
gudhi.plot_persistence_barcode(cc.persistence())
plt.show()

plt.imshow(arr_2d_np, aspect="auto", cmap="terrain")
plt.title("2D Matrix of function values")
plt.colorbar()
plt.show()

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from scipy.spatial.distance import cdist
from sklearn.neighbors import NearestNeighbors

np.random.seed(42)

# --- Generate manifold with two holes ---
def make_manifold():
    pts = []
    # Outer ring of points (main body)
    for _ in range(600):
        while True:
            x, y = np.random.uniform(-3, 3, 2)
            d1 = np.sqrt((x+1.2)**2 + y**2)
            d2 = np.sqrt((x-1.2)**2 + y**2)
            if d1 > 0.55 and d2 > 0.55 and np.sqrt(x**2+y**2) < 2.9:
                pts.append([x, y])
                break
    return np.array(pts)

pts = make_manifold()

# --- Assign topological regions by proximity to holes ---
hole1 = np.array([-1.2, 0.0])
hole2 = np.array([1.2, 0.0])

d1 = np.linalg.norm(pts - hole1, axis=1)
d2 = np.linalg.norm(pts - hole2, axis=1)
dmin = np.minimum(d1, d2)

# 3 regions: near hole1, near hole2, far from both
region = np.where(d1 < d2, 
                  np.where(d1 < 1.1, 0, 2),
                  np.where(d2 < 1.1, 1, 2))

colors_region = ['#4C72B0', '#DD8452', '#55A868']
region_labels = ['Region A (hole 1)', 'Region B (hole 2)', 'Region C (rest)']

# --- Geodesic paths ---
def straight_path(start, end, n=80):
    return np.linspace(start, end, n)

def curved_path(start, end, via, n=80):
    t = np.linspace(0, 1, n)
    # Quadratic bezier
    path = np.outer((1-t)**2, start) + np.outer(2*(1-t)*t, via) + np.outer(t**2, end)
    return path

start = np.array([-2.5, -0.3])
end   = np.array([2.5,   0.3])
via1  = np.array([0.0,  2.0])   # goes above both holes
bad_path  = straight_path(start, end)
good_path = curved_path(start, end, via1)

# ---- PLOT ----
# fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), facecolor='#FAFAFA')
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), facecolor='#FAFAFA')
plt.subplots_adjust(wspace=-0.7)

fig.suptitle('Topology-Guided Metric Learning on Manifolds', 
             fontsize=14, fontweight='bold', y=1.01, color='#1a1a1a')

for ax_idx, ax in enumerate(axes):
    ax.set_facecolor('#FAFAFA')
    ax.set_xlim(-3.2, 3.2)
    ax.set_ylim(-3.2, 3.2)
    ax.set_aspect('equal')
    ax.axis('off')

    # Draw points colored by region
    for r, c in enumerate(colors_region):
        mask = region == r
        # alpha = 0.18 if ax_idx == 0 else 0.35
        alpha = 0.7
        ax.scatter(pts[mask, 0], pts[mask, 1], c=c, s=6, alpha=alpha, zorder=1)

    # Draw holes
    for hx, hy in [(-1.2, 0), (1.2, 0)]:
        circle = plt.Circle((hx, hy), 0.52, color='#cccccc', zorder=2, linewidth=1.5,
                             fill=True, facecolor='#e8e8e8', edgecolor='#999999', linestyle='--')
        ax.add_patch(circle)
        ax.text(hx, hy, 'hole', ha='center', va='center', fontsize=7.5,
                color='#888888', style='italic')

    if ax_idx == 0:
        # Panel 1: naive geodesic (straight line through holes)
        ax.plot(bad_path[:, 0], bad_path[:, 1], color='#C0392B', lw=2.2,
                zorder=5)#, label='Naive geodesic')
        ax.annotate('', xy=bad_path[60], xytext=bad_path[55],
                    arrowprops=dict(arrowstyle='->', color='#C0392B', lw=2))
        ax.scatter([start[0], end[0]], [start[1], end[1]], 
                   c='#1a1a1a', s=60, zorder=6)
        ax.text(start[0]-0.15, start[1]-0.28, '$p$', fontsize=11, color='#1a1a1a', fontweight='bold')
        ax.text(end[0]+0.05,   end[1]-0.28,   '$q$', fontsize=11, color='#1a1a1a', fontweight='bold')
        ax.set_title('(a) Naive geodesic', fontsize=11, pad=8, color='#1a1a1a')
        # ax.text(0, -3.5, 'Geodesic ignores topological structure.', ha='center', fontsize=8, color='#555555', transform=ax.transData)
        # leg = ax.legend(loc='lower center', fontsize=9, framealpha=0.7)

    else:
        # Panel 2: topology-aware geodesic + colored regions
        for r, c in enumerate(colors_region):
            mask = region == r
            ax.scatter(pts[mask, 0], pts[mask, 1], c=c, s=8, alpha=0.55, zorder=1)

        ax.plot(good_path[:, 0], good_path[:, 1], color='#27AE60', lw=2.4,
                zorder=5)#, label='Topology-regularized geodesic')
        ax.annotate('', xy=good_path[55], xytext=good_path[50],
                    arrowprops=dict(arrowstyle='->', color='#27AE60', lw=2))
        ax.scatter([start[0], end[0]], [start[1], end[1]],
                   c='#1a1a1a', s=60, zorder=6)
        ax.text(start[0]-0.15, start[1]-0.28, '$p$', fontsize=11, color='#1a1a1a', fontweight='bold')
        ax.text(end[0]+0.05,   end[1]-0.28,   '$q$', fontsize=11, color='#1a1a1a', fontweight='bold')

        patches = [mpatches.Patch(color=c, label=l, alpha=0.7) 
                   for c, l in zip(colors_region, region_labels)]
        # patches.append(mpatches.Patch(color='#27AE60'))#, label='Topology-regularized geodesic'))
        ax.legend(handles=patches, loc='lower center', fontsize=8.5, framealpha=0.75)
        # ax.legend(handles=patches, loc='upper right', fontsize=8.5, framealpha=0.75)
        ax.set_title('(b) Persistence-regularized geodesic',
                     fontsize=11, pad=8, color='#1a1a1a')

plt.tight_layout()
plt.savefig('topo_geo_figure.png', bbox_inches='tight', dpi=200)
print("Saved.")

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

np.random.seed(42)

def make_manifold():
    pts = []
    for _ in range(700):
        while True:
            x, y = np.random.uniform(-3, 3, 2)
            d1 = np.sqrt((x+1.2)**2 + y**2)
            d2 = np.sqrt((x-1.2)**2 + y**2)
            if d1 > 0.55 and d2 > 0.55 and np.sqrt(x**2+y**2) < 2.9:
                pts.append([x, y])
                break
    return np.array(pts)

pts = make_manifold()

hole1 = np.array([-1.2, 0.0])
hole2 = np.array([1.2, 0.0])
d1 = np.linalg.norm(pts - hole1, axis=1)
d2 = np.linalg.norm(pts - hole2, axis=1)

region = np.where(d1 < d2,
                  np.where(d1 < 1.1, 0, 2),
                  np.where(d2 < 1.1, 1, 2))

metric_val = np.sin(pts[:, 0] * 0.8) * np.cos(pts[:, 1] * 0.8) + 0.3 * np.random.randn(len(pts))

colors_region = ['#4C72B0', '#DD8452', '#55A868']
region_labels = ['Region A', 'Region B', 'Region C']

tensor_params = [
    {'cx': -1.2, 'cy': -1.8, 'w': 0.55, 'h': 0.28, 'angle': 30},
    {'cx':  1.2, 'cy': -1.8, 'w': 0.28, 'h': 0.55, 'angle': 60},
    {'cx':  0.0, 'cy':  1.8, 'w': 0.50, 'h': 0.22, 'angle': 0},]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), facecolor='#FAFAFA')
plt.subplots_adjust(wspace=-0.7)
fig.suptitle('Topology-Guided Metric Estimation', fontsize=14, fontweight='bold', y=1.01, color='#1a1a1a')

for ax_idx, ax in enumerate(axes):
    ax.set_facecolor('#FAFAFA')
    ax.set_xlim(-3.2, 3.2)
    ax.set_ylim(-3.5, 3.2)
    ax.set_aspect('equal')
    ax.axis('off')

    for hx, hy in [(-1.2, 0), (1.2, 0)]:
        circle = plt.Circle((hx, hy), 0.52, zorder=3, linewidth=1.5,
                             facecolor='#e8e8e8', edgecolor='#999999', linestyle='--')
        ax.add_patch(circle)
        ax.text(hx, hy, 'hole', ha='center', va='center', fontsize=7.5,
                color='#888888', style='italic', zorder=4)

    if ax_idx == 0:
        norm = Normalize(vmin=metric_val.min(), vmax=metric_val.max())
        sc = ax.scatter(pts[:, 0], pts[:, 1], c=metric_val, cmap='plasma',
                        s=8, alpha=0.85, zorder=2, norm=norm)
        for i in range(0, len(pts), 18):
            e = mpatches.Ellipse((pts[i,0], pts[i,1]),
                                  width=0.18, height=0.09,
                                  angle=np.degrees(np.arctan2(pts[i,1], pts[i,0])),
                                  linewidth=0.6, edgecolor='white', facecolor='none',
                                  alpha=0.5, zorder=3)
            ax.add_patch(e)
        # plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, label='metric value')
        ax.set_title('(a) Pointwise metric G computed at every point',
                     fontsize=11, pad=8, color='#1a1a1a')
        # ax.text(0, -3.2, 'Expensive: one estimation per point.', ha='center',
        #         fontsize=8, color='#555555', transform=ax.transData)

    else:
        for r, c in enumerate(colors_region):
            mask = region == r
            ax.scatter(pts[mask, 0], pts[mask, 1], c=c, s=8, alpha=0.55, zorder=2)

        # for r, (tp, c) in enumerate(zip(tensor_params, colors_region)):
        #     e = mpatches.Ellipse((tp['cx'], tp['cy']),
        #                           width=tp['w']*2, height=tp['h']*2,
        #                           angle=tp['angle'],
        #                           linewidth=2, edgecolor=c,
        #                           facecolor=c, alpha=0.35, zorder=4)
        #     ax.add_patch(e)
        #     ax.text(tp['cx'], tp['cy'], '$G$', ha='center', va='center',
        #             fontsize=9, fontweight='bold', color=c, zorder=5)

        patches = [mpatches.Patch(color=c, label=l, alpha=0.7)
                   for c, l in zip(colors_region, region_labels)]
        ax.legend(handles=patches, loc='upper right', fontsize=8.5, framealpha=0.75)
        ax.set_title('(b) Region-wise metric G computed per region',
                     fontsize=11, pad=8, color='#1a1a1a')
        # ax.text(0, -3.2, 'Cheap: one estimation per topological class.', ha='center',
        #         fontsize=8, color='#555555', transform=ax.transData)

plt.tight_layout()
plt.savefig('topo_metric_figure.png', bbox_inches='tight', dpi=200)
plt.savefig('topo_metric_figure.pdf', bbox_inches='tight', dpi=200)
print("Saved.")